# Test LoRA Model on Google Colab

This notebook allows you to load and test your fine-tuned LoRA model using Google Colab's free GPU resources.

## Prerequisites
1.  **Upload your model**: Upload your `lora_model` folder to your Google Drive.
2.  **Enable GPU**: Go to `Runtime` > `Change runtime type` > Select `T4 GPU`.

In [ ]:
# @title 1. Install Dependencies
!pip install -q torch transformers peft bitsandbytes accelerate

In [ ]:
# @title 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title 3. Setup Paths
import os

# UPDATE THIS PATH to where you uploaded your 'lora_model' folder in Drive
# Example: "/content/drive/MyDrive/DataScience_Rag_Model/lora_model"
MODEL_PATH = "/content/drive/MyDrive/lora_model" 

if not os.path.exists(MODEL_PATH):
    print(f"WARNING: Path not found: {MODEL_PATH}")
    print("Please verify the path to your uploaded model folder.")
    print("Available files in Drive:")
    !ls /content/drive/MyDrive
else:
    print(f"Model path found: {MODEL_PATH}")

In [ ]:
# @title 4. Load Model & Tokenizer
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("Loading tokenizer...")
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    print("Tokenizer loaded.")
except Exception as e:
    print(f"Error loading tokenizer: {e}")

print("Loading model...")
try:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        quantization_config=quantization_config,
        device_map="auto",
        trust_remote_code=True
    )
    print("Model loaded successfully!")
except Exception as e:
    print(f"Error loading model: {e}")

In [ ]:
# @title 5. Run Inference
prompt = "What is the capital of France?" # @param {type:"string"}

messages = [
    {"role": "user", "content": prompt}
]

if tokenizer.chat_template:
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
else:
    text = prompt

print(f"Input: {text}")

inputs = tokenizer(text, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7,
    top_p=0.9
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\nModel Response:")
print(response)